# Where is the headroom over EI? Basin decomposition of static arms and luck-corrected option labels

## Description

<!-- Written 2026-11-12 before the first run (pre-registration). Design: research_log/decisions/0008 (shared reasoning with Tim). Issue #13. -->

**Question.** Two rounds of candidate-level policy improvement found no advantage over EI (api_round1, api_decoder). Decision 0008 traced this to four causes:
- a linear reward;
- label noise from our own stochastic EI;
- log regret mixing basin discovery with polishing;
- one-step deviations that the horizon washes out.

Luck-corrected switching headroom does exist with task knowledge: BBOB +66% at d = 2 (decided); the prior pilot +37%, indicative only.

This experiment asks, with exact and luck-corrected labels: **is there an advantage over EI in following a different acquisition rule for k steps, and does it come from finding the right basin?** It is the signal any policy improvement needs, measured before any learning.

**Setup** (`experimental/options.py`, `tasks/option_headroom.py`). d = 2, n_init = 4, T = 20, pfn_large, fp32.
- *Deterministic arms.* Every arm maximizes its acquisition over a candidate set that depends only on the data and a seed s: a scrambled Sobol set (1024 points) plus fixed perturbations of the incumbent (64 at each of σ = 0.01, 0.05, 0.2).
  - For a fixed s, every policy is deterministic. The tasks are noise-free, so one continuation is an exact outcome.
  - Two seeds, s ∈ {0, 1}, are two equally uninformed candidate sets. They are the **luck replicates**.
  - Arms: EI, PI(ξ = 0.01), UCB(0.995), MES, posterior mean (static_tune's portfolio without ES).
- *Reward.* Terminal log10 regret, log10(best − f\* + ε) with ε = 1e-7 (BO standard).
  - f\* comes from the strong search (16384 Sobol points plus 16×200 Adam steps), lowered by every run.
  - A 16× stronger search never improved f\* by more than 1e-6 at d = 2 (track record 2026-11-12).
- *Basins (privileged, diagnostic only).*
  - Projected Adam descent on the true f (300 steps, lr 0.003) from an incumbent gives the value of the basin it drains to.
  - "Good basin" = within δ ∈ {1, 10} PFN bins (0.001) of f\*.
  - A second setting (600 steps, lr 0.001) checks the label's robustness.
- *Part A* (split 0, 500 tasks): full runs of every arm under both seeds. Per step: the regret, and whether the incumbent is in a good basin.
- *Part B* (split 4, 800 fresh tasks, task offset 20000):
  - **Branch state:** one per task, by EI (seed 0) to a step t drawn uniformly from 0–15.
  - **Base:** EI to T.
  - **Options:** each of PI, UCB, MES and mean for k ∈ {1, 3, 5} steps, then EI to T.
  - **Nulls:** `ei@j`, j ∈ {0, 1}, = EI over a fixed random half of the candidates for k steps. Same kind of deviation, no information.
  - Everything runs under both seeds; the advantage of option o under seed s is log10 R_T(base, s) − log10 R_T(o, s), positive = better.
  - Per lane, also recorded:
    - whether the first pick differs from EI's;
    - the final incumbent's basin;
    - the belief after the option: P(y(x\*) < state incumbent) and the distance from the belief's predicted minimum to x\*.

**What to look for.**
1. **Primary** (Part B): the cross-fitted headroom H₃ of the arms.
   - Per state, choose the option with the best advantage on one seed, or none if no advantage is > 0.
   - Score that choice on the other seed, average both directions, then average over states, with a bootstrap CI over states.
   - An option that is best only by luck of the candidate set does not transfer across seeds.
2. Secondary (Part B):
   - H₁ and H₅;
   - the same for the null family, and arms − nulls: the luck scale and the luck-corrected contrast;
   - the in-sample best (oracle), biased upwards, for scale;
   - per option: the mean advantage, the share that disagree with EI, the net basin gain (P(option ends in a good basin while EI does not) − the reverse), and the belief measures against the nulls at the same k;
   - everything by t.
3. Part A (descriptive):
   - P(good basin by t) per arm and seed;
   - log-regret curves per arm;
   - how much of the differences between arms is basin discovery (the regret given a good basin vs the share in a good basin);
   - seed-to-seed spread as the luck scale.

**GO / NO-GO** (rule `analysis/options.py::option_verdict`, on the arms' H₃; worthwhile = 0.022 log10, i.e. 5% less regret).
- **GO**: the CI of H₃ is above 0 and H₃ ≥ 0.022. There is a luck-corrected, exploitable advantage over EI in switching for 3 steps. Next: can the belief find it (0008 §4 step 3)?
- **NO-GO**: the upper CI bound of H₃ is < 0.022. Even with task knowledge, 3-step switching does not beat EI beyond luck on this prior at d = 2. Next: higher d, or longer options.
- **INCONCLUSIVE**: otherwise.

**Limits.**
- Noise-free P0, d = 2, T = 20 only; d = 2 is mostly solved by EI (0007).
- Two seeds make a coarse luck correction. Cross-fitting removes selection bias, but an option whose advantage depends on the candidate set itself counts as luck.
- One branch state per task.
- Basin labels by local descent can mislead (near-equal optima, valleys from irrelevant dimensions, plateaus; 0008 §3). They are a diagnostic, never the reward.
- The headroom is privileged: an upper bound on what a belief-based switching policy can reach, not a policy.


In [ ]:
from anytime_acq.analysis import load_snapshot, paired_diff, show_header, summarize

SNAPSHOT = "results/option_headroom/<sweep_id>"  # set after `just snapshot runs/option_headroom/<sweep_id>`
snap = load_snapshot(SNAPSHOT)
show_header(snap)  # how it was run + configuration, generated from the snapshot

## Results

In [ ]:
# Aggregates use DONE members only; n_failed / n_started / n_missing say what is not in them yet.
summarize(snap.runs, "<metric>")

## Verdict

The GO / NO-GO criterion from the description, evaluated, in one line. Record it with `just decide`.